# Coastal Flooding Analysis

This notebook implements the full coastal flooding analysis pipeline for ~55 US coastal urban road networks. It ingests NOAA SLOSH Maximum of Maximums (MOM) storm surge inundation rasters for Saffir–Simpson hurricane Categories 1–5 and propagates those flood depths through each road network graph to produce:

1. **Direct exposure metrics** — total and percentage of road length inundated per hurricane category
2. **Disrupted network graphs** — edges with ≥ 1 ft (≈ 0.3 m) inundation removed and remaining edges assigned flood-adjusted travel speeds and times
3. **Indirect impact metrics** — percentage of origin–destination (O-D) pairs whose shortest path is severed, plus flood-condition route lengths and travel times

## Section 1. Import Libraries

In [ ]:
# ── Standard library ────────────────────────────────────────────────────────────
import os
import shutil
import pickle

# ── Geospatial ──────────────────────────────────────────────────────────────────
import networkx as nx
import geopandas as gpd
import rasterio as rio
import osmnx as ox
from pyproj import Transformer

# ── Data science ────────────────────────────────────────────────────────────────
import numpy as np
import math
import pandas as pd

# ── Progress bars ───────────────────────────────────────────────────────────────
from tqdm import tqdm

# ── Display configuration ───────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Set home directory
home_dir = '/Users/yiyi/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/Research/Global road network resilience/01_data' # CURA
home_dir = '/Users/yiyi/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology(2)/Research/Global road network resilience/01_data' # SCARP
# home_dir = '/Users/yiyi/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/Research/Global road network resilience/01_data' # Home

---
## Section 2 — Data Preparation

**Objective:** Identify the subset of urban road network graphs that fall within US coastal hurricane-surge zones and copy them to a dedicated working directory (`Coastal_flooding/graphs/`) for the remainder of this analysis.

**Input:** `network_polys_IDs.csv` — lookup table mapping coastal network polygons to their `net_id` values, produced by a prior spatial intersection with the NOAA SLOSH coverage footprint.

**Output:** Graph pickle files copied to `Coastal_flooding/graphs/`.  
Only networks whose `net_id` appears in the lookup table are included; all others are skipped silently.

In [ ]:
# Identify coastal network IDs from the pre-built lookup CSV.
# coastal_networkID_df maps each coastal network polygon to its integer net_id.
#
# Walk the global graph store (all_graph_flooded/) and copy any graph whose
# net_id appears in the coastal list into the coastal working directory.
# shutil.copy2 preserves file metadata (timestamps, permissions).
# Files not matching a coastal net_id are silently skipped with `continue`.
'''Housekeeping items'''
# Make a copy of networks that are affected by hurricane CAT1-5
coastal_networkID_df = pd.read_csv(home_dir + '/Coastal_flooding/network_polys_IDs.csv')

# Array of network ids
netids = coastal_networkID_df.net_id.values.astype(int)

# Copy graphs
source_folder = home_dir + '/all_graph_flooded/'
target_folder = home_dir + '/Coastal_flooding/graphs/'

# Find networks in netids that are affected by coastal flooding and make a copy to new location
for file in os.listdir(source_folder):
    if file.endswith('.pk'):
        netid = int(file.split('_')[4])
        if netid in netids:
            # make a copy
            shutil.copy2(os.path.join(home_dir,source_folder, file),
                         os.path.join(home_dir,target_folder, file))
        else:
            continue

---
## Section 3 — Attach NOAA SLOSH Inundation Values

**Objective:** Sample the NOAA SLOSH MOM storm surge inundation raster at every graph node for each hurricane category, then propagate those values to edges using a conservative (maximum) aggregation rule.

**Flood data:** NOAA SLOSH MOM (Maximum of Maximums) inundation rasters represent the highest modelled surge at each location across all SLOSH basin simulations for a given category — a worst-credible-case scenario.

**Raster CRS:** EPSG:4269 (NAD83 geographic). Node coordinates (WGS-84, EPSG:4326) are reprojected using `pyproj.Transformer` before pixel lookup.

**Two-step process:**
1. **Node annotation** — sample the integer inundation band at each node's reprojected (x, y) position; store as `CAT_{N}_value`
2. **Edge annotation** — assign each edge the maximum of its two endpoint values (`take_val` helper); a NaN endpoint is ignored so a valid endpoint value propagates

In [ ]:
# Pre-load all five NOAA SLOSH MOM rasters into memory-mapped objects.
# Opening rasters once outside the node loop avoids repeated file I/O overhead.
#
# hurricane_rasters[cat] — open rasterio DatasetReader for category N
# hurricane_bands[cat]   — numpy array of band 1 (inundation depth class)
#
# Coordinate transformer: WGS-84 lon/lat → raster CRS (EPSG:4269, NAD83).
# always_xy=True ensures (longitude, latitude) input order regardless of
# the axis order defined in the CRS authority.
# Location of graphs
graphs_dir = home_dir + '/Coastal_flooding/graphs/'

# Location of coastal flood risk maps
coastal_flooding_dir = '/Users/yiyi/Desktop/Coastal Flooding/NOAA_National_storm_surge/US_SLOSH_MOM_Inundation_20250923'

hurricane_categories = [1,2,3,4,5]

hurricane_rasters = {}
hurricane_bands = {}

for hurricane_category in hurricane_categories:
    # coastal flood risk map raster
    src = rio.open(f'/Users/yiyi/Desktop/Coastal Flooding/NOAA_National_storm_surge/US_SLOSH_MOM_Inundation_20250923/us_Category{hurricane_category}_MOM_Inundation_HIGH.tif')

    hurricane_rasters[hurricane_category] = src
    hurricane_bands[hurricane_category] = src.read(1) # load raster band
# Transform to lat lon
transformer = Transformer.from_crs(
    "EPSG:4326",      # lon/lat
    src.crs,  # EPSG:4269
    always_xy=True
)

### NOAA SLOSH MOM Inundation Value Encoding

The integer pixel values in the SLOSH MOM rasters encode inundation depth class:

| Value | Meaning |
|-------|---------|
| 1 | 0–1 ft above ground |
| 2 | 1–2 ft above ground |
| 3 | 2–3 ft above ground |
| … | … |
| 20 | 19–20 ft above ground |
| 21 | Greater than 20 ft above ground |
| 99 | Levee area — consult local officials for flood risk |
| NaN | Outside raster extent or no inundation modelled |

**Flooded edge criterion used in this notebook:** `CAT_{N}_value > 1`  
i.e. any edge with ≥ 1 ft (≈ 0.3 m) of inundation is considered impassable (consistent with the Pregnolato et al. 2017 speed–depth relationship used below).

In [ ]:
# Node-level inundation annotation loop.
#
# For each graph in the coastal set:
#   1. Load the pickle and extract the net_id from the filename.
#   2. For every node, reproject its (lon, lat) to the raster CRS.
#   3. Use rasterio's src.index() to convert projected coordinates to
#      pixel (row, col), then read the inundation class from the pre-loaded band.
#   4. IndexError is caught for nodes that fall outside the raster extent
#      (e.g., inland nodes near the raster boundary) and assigned NaN.
#   5. The value is stored as a node attribute: g.nodes[node][f'CAT_{N}_value'].
#
# Annotated graphs are saved to graphs_node_flooded/ with protocol=2
# for backwards compatibility with Python 2 readers.
for file in tqdm(os.listdir(graphs_dir)):
    if file.endswith('.pk'):
        # Read in the graph pickle file
        g = pickle.load(open(graphs_dir + file, 'rb'))
        # Network id
        net_id = int(file.split("_")[4])

        #
        for node, data in g.nodes(data=True):
            lon = data["x"]
            lat = data["y"]

            # Reproject lon/lat to raster crs
            x, y = transformer.transform(lon, lat)

            # Iterate through hurricane categories
            for hurricane_category in hurricane_categories:
                # Load hurricane raster, band
                src = hurricane_rasters[hurricane_category]
                band = hurricane_bands[hurricane_category]
                # Get raster value at node
                try:
                    row, col = src.index(x, y)
                    value = band[row, col]

                    if nodata is not None and value == nodata:
                        value = np.nan

                except IndexError:
                    value = np.nan  # node outside raster extent
                # Attach raster value to node attribute
                g.nodes[node][f"CAT_{hurricane_category}_value"] = value

        # Save the graph
        with open(home_dir + f"/Coastal_flooding/graphs_node_flooded/net_{net_id}.pk", "wb") as f:
            pickle.dump(g, f, protocol=2)

100%|██████████| 56/56 [03:11<00:00,  3.43s/it]


In [ ]:
# Edge-level inundation annotation using the take_val() helper.
#
# take_val(i, j) implements a conservative maximum rule:
#   - If both endpoints have NaN → edge value = NaN (outside flood zone)
#   - If one endpoint is NaN    → use the other endpoint's value
#   - Otherwise                 → take the larger of the two values
#
# This is conservative because an edge crossing a flooded zone is as exposed
# as its most-flooded endpoint; using the maximum avoids underestimating
# inundation for edges that span a flood boundary.
#
# Output graphs saved to graphs_node_edge_flooded/.

# Add inundation values on graph edges
def take_val(i, j):
    if math.isnan(i) and math.isnan(j):
        return float("nan")
    if math.isnan(i):
        return j
    if math.isnan(j):
        return i
    return max(i, j)

graph_flood_dir = home_dir + '/Coastal_flooding/graphs_node_flooded/'
for file in tqdm(os.listdir(graph_flood_dir)):
    if file.endswith('.pk'):
        net_id = int("".join(filter(str.isdigit, file)))
        # Read in the graph pickle file
        g = pickle.load(open(graph_flood_dir + file, 'rb'))
        for i,j,data in g.edges.data():
            for hurricane_category in hurricane_categories:
                i_val = g.nodes[i][f'CAT_{hurricane_category}_value']
                j_val = g.nodes[j][f'CAT_{hurricane_category}_value']
                val = take_val(i_val,j_val)
                g.edges[i, j, 0][f'CAT_{hurricane_category}_value'] = val
        # Save the graph
        with open(home_dir + f"/Coastal_flooding/graphs_node_edge_flooded/net_{net_id}.pk", "wb") as f:
            pickle.dump(g, f, protocol=2)

---
## Section 4 — Direct Flood Exposure

**Objective:** For each coastal network and each hurricane category, compute:
- **`CAT{N}_expo`** — total road length (m) with SLOSH inundation value > 1 (i.e., ≥ 1 ft of surge)
- **`CAT{N}_expo_perc`** — exposure as a percentage of the network's total road length

**Flooded-edge criterion:** `data['CAT_{N}_value'] > 1`  
Edges with `value == NaN` (outside raster extent) are treated as not flooded.  
Edges with `value == 1` (0–1 ft inundation) are also excluded — consistent with the speed-reduction model's impassability threshold applied in Section 4.

**Output file:** `Coastal_flooding/graph_exposure.csv`  
Columns: `net_id`, `total_length`, `CAT1_expo` … `CAT5_expo`, `CAT1_expo_perc` … `CAT5_expo_perc`

In [ ]:
# Direct exposure computation loop.
#
# For each annotated graph:
#   1. Sum all edge lengths → total_length (m)
#   2. For each hurricane category, sum lengths of edges where
#      CAT_{N}_value > 1 (at least 1 ft of surge inundation).
#      - NaN values (outside raster) are skipped with `continue`.
#      - KeyError is caught for edges missing the attribute (defensive coding).
#   3. Assemble results into a DataFrame and compute exposure percentages.
#
# Note: exposure percentage can theoretically exceed 100 % only if edge
# lengths are inconsistent — a sanity check on the output is advisable.
hurricane_categories = [1,2,3,4,5]

net_id_col = []
total_length_col = []
cat1_expo_col = []
cat2_expo_col = []
cat3_expo_col = []
cat4_expo_col = []
cat5_expo_col = []
cat_lsts = [cat1_expo_col, cat2_expo_col, cat3_expo_col, cat4_expo_col, cat5_expo_col]
# 
for file in tqdm(os.listdir(home_dir + "/Coastal_flooding/graphs_node_edge_flooded/")):

    net_id = int("".join(filter(str.isdigit, file)))
    net_id_col.append(net_id)
    g = pickle.load(open(home_dir + "/Coastal_flooding/graphs_node_edge_flooded/"+ file, 'rb'))
    
    total_length = 0
    for i,j,data in g.edges.data():
        total_length += data["length"]
    total_length_col.append(total_length)

    for hurricane_category in hurricane_categories:
        
        exposure_length = 0
        
        for i,j,data in g.edges.data():

            try:

                if math.isnan(data[f"CAT_{hurricane_category}_value"]):
                    continue
                elif data[f"CAT_{hurricane_category}_value"]>1:
                    exposure_length += data["length"]
            except KeyError:
                continue

        cat_lsts[hurricane_category-1].append(exposure_length)
    
exposure_data = {
    "net_id": net_id_col,
    "total_length" : total_length_col,
    "CAT1_expo" : cat1_expo_col,
    "CAT2_expo" : cat2_expo_col,
    "CAT3_expo" : cat3_expo_col,
    "CAT4_expo" : cat4_expo_col,
    "CAT5_expo" : cat5_expo_col
}

exposure_data_df = pd.DataFrame.from_dict(exposure_data)

exposure_data_df.head(3)

# Calculate exposure percentage
for hurricane_category in hurricane_categories:
    exposure_data_df[f'CAT{hurricane_category}_expo_perc'] = 100*exposure_data_df[f'CAT{hurricane_category}_expo']/exposure_data_df['total_length']
    
exposure_data_df.to_csv(home_dir + "/Coastal_flooding/graph_exposure.csv")

100%|██████████| 55/55 [01:00<00:00,  1.09s/it]


---
## Section 5 — Indirect Network Impact

**Objective:** Quantify the indirect consequences of coastal flooding — specifically, how many origin–destination (O-D) pairs lose their shortest path and by how much travel time and distance increase for pairs that can still be routed.

This section has four sub-steps:
1. **Copy baseline O-D files** — reuse dry/wet routing CSVs from the 30 cm pluvial analysis as the no-flood reference
2. **Build disrupted graphs** — remove flooded edges and assign surge-adjusted travel speeds using the Pregnolato et al. (2017) speed–depth relationship
3. **Handle network 2091 separately** — this network lacks OSM speed tags and requires manual road-class speed overrides
4. **Run O-D simulations** — Dijkstra shortest-path routing on each disrupted graph; record route, length, and travel time for all O-D pairs across all five categories

### 5.1 Copy Baseline O-D Routing Files

The dry-condition O-D routing results from the global 30 cm pluvial/fluvial analysis serve as the no-flood baseline. Only files whose `net_id` matches a coastal network are copied to `Coastal_flooding/Dry_wet_routing_30cm/`.

In [ ]:
# Copy baseline O-D routing CSVs for coastal networks.
# Source: the full global 30 cm dry/wet routing results directory (2576 networks).
# Destination: Coastal_flooding/Dry_wet_routing_30cm/ (55 coastal networks only).
# Only files whose numeric net_id appears in the coastal exposure DataFrame are copied;
# all others are silently skipped.

# Move original dry wet simulation results files to new location (US coastal networks only)
exposure_df = pd.read_csv(home_dir + '/Coastal_flooding/graph_exposure.csv', index_col=0)
net_ids = list(exposure_df.net_id)

for file in os.listdir(home_dir + '/OD_routing_simulation_results/2576_dry_wet_OD_routing_30cm/'):
    if file.endswith('.csv'):
        net_id = int("".join(filter(str.isdigit, file)))
        if net_id in net_ids:
            # Copy the file to new location
            shutil.copy(home_dir + '/OD_routing_simulation_results/2576_dry_wet_OD_routing_30cm/' + file,
                        home_dir + '/Coastal_flooding/Dry_wet_routing_30cm/')

### 5.2 Build Disrupted Graphs with Flood Travel Speed and Travel Time

For each coastal network and each hurricane category, a disrupted copy of the graph is constructed by:

1. **Removing flooded edges** — any edge with `CAT_{N}_value > 1` (≥ 1 ft of surge) is deleted from the graph copy. Edges with NaN values are retained (assumed unaffected).
2. **Adding baseline speeds** — `ox.routing.add_edge_speeds()` assigns posted/inferred speed limits; `ox.routing.add_edge_travel_times()` computes travel time from speed × length.
3. **Overriding with flood speed** — remaining edges near the flood zone receive a surge-adjusted travel speed derived from the Pregnolato et al. (2017) speed–inundation-depth relationship:  
   `v = 0.0009w² − 0.5529w + 86.9448` (km/h), where w = water depth in mm.  
   At the impassability threshold of 1 ft (304.8 mm), this yields **v ≈ 2.03 km/h**.  
   The effective edge speed is `min(flood_speed, baseline_speed)` to ensure the flood speed only ever reduces — never increases — travel time.

**Output:** Five disrupted graph pickles per network, saved to `graphs_disrupted/CAT{N}/net_{net_id}_disrupted_speed_time.pk`

In [ ]:
# Disrupted graph generation loop (all networks except 2091).
#
# For each graph file in graphs_node_edge_flooded/:
#   For each hurricane category N in [1, 2, 3, 4, 5]:
#     1. Copy the full graph to g_disrupted.
#     2. Remove edges where CAT_{N}_value is a valid number (i.e., not NaN).
#        Rationale: NaN means the edge is outside the SLOSH raster extent —
#        treated as not flooded. Any non-NaN value ≥ 1 means ≥ 1 ft of surge.
#     3. Add OSMnx-derived baseline speeds and travel times to remaining edges.
#     4. For each remaining edge, compute flood-adjusted speed and time:
#        flood_speed_ms = 2.03 m/s (speed at 1 ft depth, Pregnolato et al. 2017)
#        cat_speed_ms   = min(flood_speed_ms, baseline_speed_ms)
#        cat_time_s     = edge_length / cat_speed_ms
#     5. Store CAT{N}_speed_ms and CAT{N}_time_s as edge attributes.
#     6. Pickle the disrupted graph.
#
# Network 2091 is skipped here and handled separately in the next cell
# because its OSM data lacks reliable speed tags.

# Generate disrupted graphs with adjusted travel time and travel speed
hurricane_categories = [1,2,3,4,5]

for graph_file in tqdm(os.listdir(home_dir + '/Coastal_flooding/graphs_node_edge_flooded/')):
    if graph_file.endswith('.pk'):
        g = pickle.load(open(home_dir + "/Coastal_flooding/graphs_node_edge_flooded/"+ graph_file, 'rb'))
        net_id = int("".join(filter(str.isdigit, graph_file)))
        if net_id == 2091:
            continue
        else:
            # Iterate through hurricane categories [1,2,3,4,5]
            for cat in hurricane_categories:
                # Remove edges with 1 foot inundation
                g_disrupted = g.copy() # initiate the disrupted graph
                for i,j,data in g.edges.data():
                    try:
                        if math.isnan(data[f"CAT_{cat}_value"]):  # if there is less than 1 foot inundation
                            continue
                        else:
                            # remove edge since there is more than 1 foot flood inundation
                            g_disrupted.remove_edge(i,j)
                    except KeyError:
                        continue
                # calculate normal/baseline travel time and travel speed
                g_disrupted = ox.routing.add_edge_speeds(g_disrupted)
                g_disrupted = ox.routing.add_edge_travel_times(g_disrupted)

                # Add flood travel speed and travel time
                # Travel speed as a function of travel time:
                # v = 0.0009w^2 - 0.5529w + 86.9448 (Pregnolato et al 2017)
                # v: km/h;   w:mm
                # When w is 1 foot or 304.8mm, v= 2.03 km/h which is 0.564 m/s
                for i,j,data in g_disrupted.edges.data():
                    # baseline travel speed
                    baseline_speed_kph = data['speed_kph']
                    baseline_speed_ms = baseline_speed_kph/3.6
                    flood_speed_ms = 2.03
                    # Calculate flood speed (m/s) and time (s)
                    cat_speed_ms = min(flood_speed_ms, baseline_speed_ms)
                    cat_time_s = data['length']/cat_speed_ms
                    # Add flood speed and time
                    g_disrupted[i][j][0][f'CAT{cat}_speed_ms'] = cat_speed_ms
                    g_disrupted[i][j][0][f'CAT{cat}_time_s'] = cat_time_s
                
                # Save disrupted graph
                with open(home_dir + f"/Coastal_flooding/graphs_disrupted/CAT{cat}/net_{net_id}_disrupted_speed_time.pk", "wb") as f:
                    pickle.dump(g_disrupted, f, protocol=2)

100%|██████████| 55/55 [13:10<00:00, 14.38s/it]


### 5.3 Special Case — Network 2091

Network 2091 is excluded from the main loop because its OpenStreetMap data does not contain reliable `maxspeed` tags. Without them, `ox.routing.add_edge_speeds()` would assign a single fallback speed to all edges, which is unrealistic for an urban network with a mix of road types.

**Solution:** Manually specify a `hwy_speeds` dictionary with road-class-appropriate speed limits (km/h), plus a `fallback=50` for any unrecognised highway type. All other logic (flood-edge removal, Pregnolato speed override) is identical to the main loop.

In [ ]:
# Special processing for network 2091 — manual speed overrides.
#
# hwy_speeds provides class-specific speed limits (km/h) for the most common
# OSM highway types. Values follow standard speed-limit conventions for the
# US urban network context (motorway=120, trunk=100, primary=90, etc.).
# fallback=50 km/h applies to any highway type not listed (e.g., 'residential',
# 'service', 'unclassified').
#
# All other processing steps (flood edge removal, Pregnolato speed correction,
# graph serialisation) are identical to the main loop.
net_id = 2091
g = pickle.load(open(home_dir + f"/Coastal_flooding/graphs_node_edge_flooded/net_{net_id}.pk", 'rb'))
# Iterate through hurricane categories [1,2,3,4,5]
for cat in hurricane_categories:
    # Remove edges with 1 foot inundation
    g_disrupted = g.copy() # initiate the disrupted graph
    for i,j,data in g.edges.data():
        try:
            if math.isnan(data[f"CAT_{cat}_value"]):  # if there is less than 1 foot inundation
                continue
            else:
                # remove edge since there is more than 1 foot flood inundation
                g_disrupted.remove_edge(i,j)
        except KeyError:
            continue
    # calculate normal/baseline travel time and travel speed

    hwy_speeds = { # km/h
    'motorway': 120,
    'motorway_link': 120,
    'trunk': 100,
    'trunk_link': 100,
    'primary': 90,
    'primary_link': 90,
    'secondary': 80,
    'secondary_link': 80,
    'tertiary': 60,
    'tertiary_link': 60}
    
    g_disrupted = ox.routing.add_edge_speeds(g_disrupted, hwy_speeds=hwy_speeds, fallback=50)
    g_disrupted = ox.routing.add_edge_travel_times(g_disrupted)

    # Add flood travel speed and travel time
    # Travel speed as a function of travel time:
    # v = 0.0009w^2 - 0.5529w + 86.9448 (Pregnolato et al 2017)
    # v: km/h;   w:mm
    # When w is 1 foot or 304.8mm, v= 2.03 km/h which is 0.564 m/s
    for i,j,data in g_disrupted.edges.data():
        # baseline travel speed
        baseline_speed_kph = data['speed_kph']
        baseline_speed_ms = baseline_speed_kph/3.6
        flood_speed_ms = 2.03
        # Calculate flood speed (m/s) and time (s)
        cat_speed_ms = min(flood_speed_ms, baseline_speed_ms)
        cat_time_s = data['length']/cat_speed_ms
        # Add flood speed and time
        g_disrupted[i][j][0][f'CAT{cat}_speed_ms'] = cat_speed_ms
        g_disrupted[i][j][0][f'CAT{cat}_time_s'] = cat_time_s
    
    # Save disrupted graph
    with open(home_dir + f"/Coastal_flooding/graphs_disrupted/CAT{cat}/net_{net_id}_disrupted_speed_time.pk", "wb") as f:
        pickle.dump(g_disrupted, f, protocol=2)

### 5.4 Add Baseline Travel Speed and Time to Unflooded Graphs

Before running the O-D simulations, the baseline (unflooded) graph for each network must also have `speed_kph` and `travel_time` edge attributes so that baseline routing can use the same `weight` keyword as the flooded graphs.  

Graphs are loaded from `graphs_node_edge_flooded/` (which has inundation attributes but no speed data), enriched with OSMnx speed/time utilities, then saved to `graphs_speed_time/`.

In [148]:
for file in os.listdir(home_dir + '/Coastal_flooding/Dry_wet_routing_30cm/'):
    if file.endswith('.csv'):
        net_id = int("".join(filter(str.isdigit, file)))
        #
        g_baseline = pickle.load(open(home_dir + f'/Coastal_flooding/graphs_node_edge_flooded/net_{net_id}.pk', 'rb'))
        g_baseline = ox.routing.add_edge_speeds(g_baseline)
        g_baseline = ox.routing.add_edge_travel_times(g_baseline)
        # Save disrupted graph
        with open(home_dir + f"/Coastal_flooding/graphs_speed_time/net_{net_id}_speed_time.pk", "wb") as f:
            pickle.dump(g_baseline, f, protocol=2)

### 5.5 O-D Routing Simulation on Disrupted Networks

**Objective:** For each O-D pair in the baseline routing DataFrame, attempt to find a shortest path on the CAT 1–5 disrupted graphs and record the result.  

**Algorithm:** Dijkstra's shortest path (`nx.shortest_path`) weighted by `CAT{N}_time_s` (flood-adjusted travel time in seconds) for disrupted conditions, and `travel_time` for the baseline condition.

**Failure modes captured:**
- `nx.NetworkXNoPath` — no path exists because the O or D node is disconnected (recorded as NaN for route, length, and time)
- O or D node not in disrupted graph — can occur when a node sits on a removed edge that was the only connection; also recorded as NaN

**Resume logic:** The `visited` list tracks already-processed net_ids so the loop can be safely interrupted and restarted without reprocessing completed networks.

**Output:** One CSV per network — `OD_simulation_result/G_{net_id}_cat_routing.csv` — with the original dry/wet columns plus six new columns per category: `OD_route_CAT_{N}`, `OD_length_CAT_{N}`, `OD_time_CAT_{N}`, and three baseline equivalents (`OD_route_baseline`, `OD_length_baseline`, `OD_time_baseline`).

In [ ]:
# O-D routing simulation loop.
#
# Outer loop: iterate over coastal network CSVs.
#   - Skip if net_id already in `visited` (resume support).
#   - Load the original dry/wet routing DataFrame as the O-D pair source.
#
# Inner loop 1 — five hurricane categories:
#   - Load the pre-built disrupted graph for this category.
#   - For each O-D pair:
#       a. Check both O and D nodes are present in the disrupted graph.
#       b. Attempt nx.shortest_path weighted by CAT{N}_time_s.
#       c. Sum edge-level CAT{N}_time_s and 'length' along the route.
#       d. Catch nx.NetworkXNoPath → append NaN for all three metrics.
#   - Append three new columns to og_dry_wet_df.
#
# Inner loop 2 — baseline (no-flood) condition:
#   - Load the baseline graph with OSMnx travel_time attributes.
#   - Route all O-D pairs using weight='travel_time' (Dijkstra).
#   - Baseline routing is assumed always successful (no NaN handling needed
#     because the baseline graph is the fully connected unflooded network).
#
# Final step: save the augmented DataFrame to OD_simulation_result/.
hurricane_categories = [1,2,3,4,5]

# Processed files
visited = []
for item in os.listdir(home_dir + f'/Coastal_flooding/OD_simulation_result/'):
    visited.append(int("".join(filter(str.isdigit, item))))

for file in tqdm(os.listdir(home_dir + '/Coastal_flooding/Dry_wet_routing_30cm/')):
    if file.endswith('.csv'):
        # Read the original dry wet simulation files
        og_dry_wet_df = pd.read_csv(home_dir + '/Coastal_flooding/Dry_wet_routing_30cm/' + file, index_col=0)
        net_id = int("".join(filter(str.isdigit, file)))

        if net_id in visited: # if visited
            continue

        else:

            # Iterate through 5 hurricane categories
            for cat in hurricane_categories:
                # Load the corresponding disrupted graph
                g_disrupted = pickle.load(open(home_dir + f"/Coastal_flooding/graphs_disrupted/CAT{cat}/net_{net_id}_disrupted_speed_time.pk", 'rb'))
                
                OD_route_cat_lst = []
                OD_length_cat_lst = []
                OD_time_cat_lst = []
                
                # iterate through OD pairs
                for row in range(og_dry_wet_df.shape[0]):
                    # Initialize
                    OD_route_cat = np.nan
                    OD_length_cat = np.nan
                    OD_time_cat = np.nan
                    # Get the ids of O and D
                    O_id = og_dry_wet_df.iloc[row]['O_node_id']
                    D_id = og_dry_wet_df.iloc[row]['D_node_id']
                    # Check if the both O and D are still in the disrupted graph
                    if O_id in g_disrupted.nodes and D_id in g_disrupted.nodes:
                        try:
                            # Find route between O and D
                            OD_route_cat = nx.shortest_path(g_disrupted, O_id, D_id, weight=f'CAT{cat}_time_s') # route

                            # Route travel distance and travel time
                            # Initialize totals
                            route_time = 0
                            route_length = 0
                            # Iterate over consecutive nodes in the route
                            for u, v in zip(OD_route_cat[:-1], OD_route_cat[1:]):
                                edge_data = g_disrupted.get_edge_data(u, v)
                                # Some edges may have multiple entries (MultiDiGraph)
                                # Take the first one
                                if isinstance(edge_data, dict) and 0 in edge_data:
                                    edge_attr = edge_data[0]
                                else:
                                    edge_attr = edge_data
                                    
                                route_time += edge_attr.get(f'CAT{cat}_time_s', 0)     # in seconds 
                                route_length += edge_attr.get('length', 0)             # in meters
                            
                        except nx.NetworkXNoPath:
                            OD_route_cat_lst.append(np.nan)
                            OD_length_cat_lst.append(np.nan)
                            OD_time_cat_lst.append(np.nan)
                            continue
                    
                    # Add result to list    
                    OD_route_cat_lst.append(OD_route_cat)
                    OD_length_cat_lst.append(route_length)
                    OD_time_cat_lst.append(route_time)
                    
                # Add simulation results as additional columns
                og_dry_wet_df[f'OD_route_CAT_{cat}'] = OD_route_cat_lst
                og_dry_wet_df[f'OD_length_CAT_{cat}'] = OD_length_cat_lst
                og_dry_wet_df[f'OD_time_CAT_{cat}'] = OD_time_cat_lst

            # Travel route, distance and time under baseline scenario
            # load graph
            g_baseline = pickle.load(open(home_dir + f'/Coastal_flooding/graphs_speed_time/net_{net_id}_speed_time.pk', 'rb'))

            OD_route_baseline_lst = []
            OD_length_baseline_lst = []
            OD_time_baseline_lst = []

            # iterate through OD pairs
            for row in range(og_dry_wet_df.shape[0]):
                # Initialize
                OD_route_baseline = np.nan
                OD_length_baseline = np.nan
                OD_time_baseline = np.nan
                # Get the ids of O and D
                O_id = og_dry_wet_df.iloc[row]['O_node_id']
                D_id = og_dry_wet_df.iloc[row]['D_node_id']

                OD_route_baseline = nx.shortest_path(g_baseline, O_id, D_id, weight='travel_time') # route
                # Route travel distance and travel time
                # Initialize totals
                route_time = 0
                route_length = 0
                # Iterate over consecutive nodes in the route
                for u, v in zip(OD_route_baseline[:-1], OD_route_baseline[1:]):
                    edge_data = g_baseline.get_edge_data(u, v)
                    # Some edges may have multiple entries (MultiDiGraph)
                    # Take the first one
                    if isinstance(edge_data, dict) and 0 in edge_data:
                        edge_attr = edge_data[0]
                    else:
                        edge_attr = edge_data
                        
                    route_time += edge_attr.get('travel_time', 0)     # in seconds 
                    route_length += edge_attr.get('length', 0)        # in meters
                
                OD_route_baseline_lst.append(OD_route_baseline)
                OD_length_baseline_lst.append(route_length)
                OD_time_baseline_lst.append(route_time)
            # Add simulation results as additional columns
            og_dry_wet_df['OD_route_baseline'] = OD_route_baseline_lst
            og_dry_wet_df['OD_length_baseline'] = OD_length_baseline_lst
            og_dry_wet_df['OD_time_baseline'] = OD_time_baseline_lst

            # Save file
            og_dry_wet_df.to_csv(home_dir + f'/Coastal_flooding/OD_simulation_result/G_{net_id}_cat_routing.csv')

100%|██████████| 55/55 [2:10:47<00:00, 142.68s/it]  


In [ ]:
# Routing failure summary — compute percentage of failed O-D pairs per network.
#
# A failed O-D pair is one where OD_route_CAT_{N} == NaN (no path found).
# isna().mean() * 100 gives the percentage of NaN entries in each route column.
#
# Output columns are renamed from 'OD_route_CAT_{N}' to 'OD_perc_fail_CAT{N}'
# for clarity in the merged analysis file.
#
# Result saved to: Coastal_flooding/graph_percent_failed_routes.csv

# Percentage failed routes summary
routing_results_dir = home_dir + '/Coastal_flooding/OD_simulation_result/'

cols_of_interest = [
    "OD_route_CAT_1",
    "OD_route_CAT_2",
    "OD_route_CAT_3",
    "OD_route_CAT_4",
    "OD_route_CAT_5"
]

summary_rows = []
for file in tqdm(os.listdir(routing_results_dir)):
    if file.endswith('.csv'):
        net_id = int("".join(filter(str.isdigit, file)))
        routing_df = pd.read_csv(routing_results_dir + file, index_col=0)
        # percentage of nans, aka percentage of failed routes
        nan_pct = routing_df[cols_of_interest].isna().mean() * 100

        row = {"network_id": net_id}
        row.update(nan_pct.to_dict())

        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.rename(columns=lambda c: c.replace("OD_route_CAT_", "OD_perc_fail_CAT"))
summary_df.to_csv(home_dir + '/Coastal_flooding/graph_percent_failed_routes.csv')


100%|██████████| 55/55 [00:01<00:00, 31.67it/s]


---
## Section 6 — Output Consolidation

**Objective:** Merge the direct exposure table and the routing failure summary into a single analysis-ready CSV.

**Join key:** `net_id` (exposure table) ↔ `network_id` (routing failure table) — both identify the same urban road network.

**Output file:** `Coastal_flooding/graph_exposure_percent_failed_routes_merged.csv`  
This file is the primary input for the coastal flooding visualisation notebook (Section 9 of `04_Visualization.ipynb`).

**Final column schema:**

| Column | Description |
|--------|-------------|
| `net_id` | Urban road network identifier |
| `total_length` | Total road length (m) |
| `CAT{1–5}_expo` | Inundated road length (m) per hurricane category |
| `CAT{1–5}_expo_perc` | Inundated road length as % of total |
| `OD_perc_fail_CAT{1–5}` | % of O-D pairs with no feasible route under surge |

In [ ]:
# Merge direct exposure and routing failure summary tables.
# Left join on net_id / network_id — all exposure records are retained;
# routing failure records without a matching exposure entry are dropped.
# The merged DataFrame is immediately saved; no further transformations are applied
# here — all downstream analysis and visualisation is handled in notebook 04.

exposure_df = pd.read_csv(home_dir + '/Coastal_flooding/graph_exposure.csv', index_col=0)
percent_fail_df = pd.read_csv(home_dir + '/Coastal_flooding/graph_percent_failed_routes.csv', index_col=0)

merged_df = exposure_df.merge(percent_fail_df, left_on='net_id', right_on='network_id')
merged_df.to_csv(home_dir + '/Coastal_flooding/graph_exposure_percent_failed_routes_merged.csv')